In [ ]:
# 
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
import config
import functions as func
import functions_data as dfunc
import mmpm as mmpm_module
import time
import matplotlib as mpl
# ================== 配置 ==================
model_dir = '/root/'
checkpoint_epoch = 150 #-> 
save_root = os.path.join(model_dir, '3T_12echo_150epoch')
os.makedirs(save_root, exist_ok=True)

#
id_tau = [0, 1, 2, 3, 4, 5]
rescale_para = 400.0
signed = 0
type_task = 'T2mapping'
file_name = '/root/autodl-tmp//test.tfrecords'

# ================== ==================
dataset_test, dataset_test_size = dfunc.load_test_data(
    file_name=file_name,
    id_tau=id_tau,
    rescale_para=rescale_para,
    type_task=type_task,
    signed=signed
)
batch_size_test = 1
dataset_test = dataset_test.batch(batch_size_test, drop_remainder=False)

# ================================
Nc = len(id_tau)
Ns, Nk, Nt = 10, 5, 1
ini_lam, ini_rho = 0.001, 0.1
Np, sep, f = 2, 0, 3

mmpm_net = mmpm_module.MMPM_Net(
    Ns=Ns, Nk=Nk, Nt=Nt, Np=Np,
    signed=signed, f=f,
    ini_lam=ini_lam, ini_rho=ini_rho,
    test_mode=True, name='MMPM_Net'
)

inpt_b = tf.keras.layers.Input(shape=(None, None, Nc), name='images')
inpt_te = tf.keras.layers.Input(shape=(Nc,), name='tes')
para_map = mmpm_net([inpt_b, inpt_te])
model = tf.keras.Model(inputs=[inpt_b, inpt_te], outputs=para_map)

# ===================================
if checkpoint_epoch is None:
    last = func.findLastCheckpoint(save_dir=model_dir)
else:
    last = checkpoint_epoch

if last > 0:
    ckpt = os.path.join(model_dir, f'model_{last:03d}.h5')
else:
    files = sorted([f for f in os.listdir(model_dir) if f.endswith('.h5')])
    ckpt = os.path.join(model_dir, files[-1]) if files else None

if ckpt:
    print('Load weights from', ckpt)
    model.load_weights(ckpt)
else:
    raise FileNotFoundError(f'No checkpoint found in {model_dir}')

# ====================================
def to_numpy(t):
    if t is None:
        return None
    if isinstance(t, tf.Tensor):
        return t.numpy()
    return np.array(t)

# ====================================
ch = 1
results = []
best_metrics = []

for case_idx, elem in enumerate(dataset_test):
    print(f"\n======== 开始推理 Case {case_idx} ========")

    (imgs, tau), maps = elem
    imgs_np = to_numpy(imgs)
    tau_np  = to_numpy(tau)
    maps_np = to_numpy(maps)

    if imgs_np.ndim == 3:
        imgs_np = imgs_np[np.newaxis, ...]
    if tau_np is None:
        tau_np = np.tile(np.array(id_tau, dtype=np.float32)[None, :], (imgs_np.shape[0], 1))

    #
    start_time = time.time()
    y_pred = model.predict([imgs_np, tau_np], verbose=0)
    total_infer_time = time.time() - start_time
    print(f"Case {case_idx} 总推理耗时: {total_infer_time:.3f} 秒")

    y_pred = np.array(y_pred)
    if y_pred.ndim == 5 and y_pred.shape[0] == imgs_np.shape[0]:
        y_pred = np.transpose(y_pred, (1, 0, 2, 3, 4))

    stages = y_pred.shape[0]
    np.save(os.path.join(save_root, f'case{case_idx}_pred_allstages.npy'), y_pred)

    gt0 = maps_np[0] if maps_np is not None else np.zeros((y_pred.shape[2], y_pred.shape[3], y_pred.shape[4]))
    gt_ch = gt0[..., ch]

    per_case_nrmse, per_case_ssim, per_case_psnr = [], [], []

    if maps_np is not None:
        stage_times = []
        for s in range(stages):
            stage_start = time.time()
            pred_s = y_pred[s, 0, ...]
            y_true_ch = gt0[..., ch][np.newaxis, ..., np.newaxis]
            y_pred_ch = pred_s[..., ch][np.newaxis, ..., np.newaxis]

            nrmse_val = float(func.NRMSE_metric(y_true_ch, y_pred_ch).numpy())
            ssim_val  = func.SSIM_metric(y_true_ch, y_pred_ch)
            psnr_val  = func.PSNR_metric(y_true_ch, y_pred_ch)
            stage_end = time.time()

            per_case_nrmse.append(nrmse_val)
            per_case_ssim.append(ssim_val)
            per_case_psnr.append(psnr_val)
            stage_times.append(stage_end - stage_start)

            results.append({
                "Case": case_idx,
                "Stage": s,
                "Channel": ch,
                "NRMSE": nrmse_val,
                "SSIM": ssim_val,
                "PSNR(dB)": psnr_val
            })

            print(f'Case {case_idx}, Stage {s:02d} => '
                  f'NRMSE={nrmse_val:.4f}, SSIM={ssim_val:.4f}, PSNR={psnr_val:.2f} dB, '
                  f'StageTime={stage_times[-1]:.3f}s')

        min_idx = int(np.argmin(per_case_nrmse))
        best_time = stage_times[min_idx]
        best_metrics.append({
            "Case": case_idx,
            "Best_Stage": int(min_idx),
            "Best_NRMSE": float(per_case_nrmse[min_idx]),
            "Best_SSIM": float(per_case_ssim[min_idx]),
            "Best_PSNR(dB)": float(per_case_psnr[min_idx]),
            "Inference_Time(s)": best_time
        })
        print(f"Case {case_idx} 最佳 stage={min_idx}, NRMSE={per_case_nrmse[min_idx]:.4f}, "
              f"SSIM={per_case_ssim[min_idx]:.4f}, PSNR={per_case_psnr[min_idx]:.2f}, "
              f"推理耗时={best_time:.3f}s")

# ================== =================
df = pd.DataFrame(results)
csv_path = os.path.join(save_root, "metrics_results.csv")
df.to_csv(csv_path, index=False)
print(f"All metrics saved to {csv_path}")

# ====================================
if len(best_metrics) > 0:
    df_best = pd.DataFrame(best_metrics)

    # 
    avg_row = {
        "Case": "Average",
        "Best_Stage": "-",
        "Best_NRMSE": df_best["Best_NRMSE"].mean(),
        "Best_SSIM": df_best["Best_SSIM"].mean(),
        "Best_PSNR(dB)": df_best["Best_PSNR(dB)"].mean(),
        "Inference_Time(s)": df_best["Inference_Time(s)"].mean()
    }
    std_row = {
        "Case": "Std",
        "Best_Stage": "-",
        "Best_NRMSE": df_best["Best_NRMSE"].std(),
        "Best_SSIM": df_best["Best_SSIM"].std(),
        "Best_PSNR(dB)": df_best["Best_PSNR(dB)"].std(),
        "Inference_Time(s)": df_best["Inference_Time(s)"].std()
    }
    df_best = pd.concat([df_best, pd.DataFrame([avg_row]), pd.DataFrame([std_row])], ignore_index=True)

    best_csv_path = os.path.join(save_root, "best_metrics_per_case.csv")
    df_best.to_csv(best_csv_path, index=False)
    print("\n======== 每个 case 最佳指标（含均值和标准差） ========")
    print(df_best)
    print(f"\n已保存至 {best_csv_path}")
else:
    print("未计算到任何最佳指标。")

# ====================================
print("\n======== 保存所有 GT 和 Stage 图像（每个 case 一个子文件夹） ========")

dataset_iter = iter(dataset_test.unbatch().batch(1))

for case_idx, case_info in enumerate(best_metrics):
    print(f"\n---- 保存 Case {case_idx} 图像 ----")

    # =================
    case_save_dir = os.path.join(save_root, f'case{case_idx}')
    os.makedirs(case_save_dir, exist_ok=True)

    best_stage = case_info["Best_Stage"]

    imgs_elem, maps_elem = next(dataset_iter)
    gt_case = to_numpy(maps_elem)[0]
    gt_ch = gt_case[..., ch]

    # 
    fg_mask = (gt_ch > 0).astype(np.float32)

    
    gt_vis = gt_ch * fg_mask

    # ---------- --------
    save_gt_path = os.path.join(case_save_dir, f'case{case_idx}_GT.png')
    plt.figure(figsize=(4,4))
    plt.imshow(gt_vis, cmap='jet')
    plt.axis('off')
    plt.savefig(save_gt_path, dpi=150, bbox_inches='tight', pad_inches=0)
    plt.close()
    print(f"  ✅ 保存 GT: {save_gt_path}")

    # -------------------
    save_colorbar_quartiles(
        'jet',
        gt_vis,
        os.path.join(case_save_dir, f'case{case_idx}_GT_colorbar.png')
    )

    # ---------- --------
    npy_path = os.path.join(save_root, f'case{case_idx}_pred_allstages.npy')
    if not os.path.exists(npy_path):
        print(f"[警告] 未找到 {npy_path}，跳过本 case")
        continue

    preds = np.load(npy_path)

    #
    if preds.ndim == 5:
        preds = preds[:, 0, ...]
    stages = preds.shape[0]

    # -------------------
    for s in range(stages):
        pred_s = preds[s, ..., ch]
        pred_vis = pred_s * fg_mask

        # ---------------
        save_stage_path = os.path.join(case_save_dir, f'case{case_idx}_stage{s}.png')
        plt.figure(figsize=(4,4))
        plt.imshow(pred_vis, cmap='jet')
        plt.axis('off')
        plt.savefig(save_stage_path, dpi=150, bbox_inches='tight', pad_inches=0)
        plt.close()
        print(f"  ▸ 保存 Stage {s}: {save_stage_path}")

        save_colorbar_quartiles(
            'jet',
            pred_vis,
            os.path.join(case_save_dir, f'case{case_idx}_stage{s}_colorbar.png')
        )

        # --------------
        error_map = np.abs(gt_ch - pred_s) * fg_mask
        save_err_path = os.path.join(case_save_dir, f'case{case_idx}_stage{s}_error.png')
        plt.figure(figsize=(4,4))
        plt.imshow(error_map, cmap='jet')
        plt.axis('off')
        plt.savefig(save_err_path, dpi=150, bbox_inches='tight', pad_inches=0)
        plt.close()
        print(f"  ▸ 保存 Stage {s} Error: {save_err_path}")

       